# AstroFoundation — Notebook 06: End-to-End System Results & Agent Demonstration
### Tri-Modal Neural Inference $\rightarrow$ LangGraph ReAct Verification $\rightarrow$ Scientific Report Generation

This notebook demonstrates the complete live AstroFoundation workflow:
1. Multi-modal forward pass with energy-based out-of-distribution anomaly scoring.
2. Autonomous LangGraph agent queries to SIMBAD, NED, and NASA ADS.
3. Pydantic-validated Astronomical Report generation.

In [1]:
from src.models.full_model import AstroFoundationModel
from src.agent import run_astro_agent
from src.reporting import AstronomicalReportGenerator

print('All AstroFoundation system components imported successfully.')

## 1. Analyze Confirmed Exoplanet Transit (Kepler-10)

In [2]:
agent_out = run_astro_agent(
    target_id='Kepler-10',
    initial_class='exoplanet_transit',
    confidence=0.96,
    energy_score=-0.22,
    is_anomaly=False
)

print(f'Target: {agent_out.target_id}')
print(f'Final Verdict: {agent_out.final_classification}')
print(f'Consensus: {agent_out.consensus_status}')
print(f'Databases Queried: {agent_out.catalogs_queried}')
print(f'Reasoning Trace Steps: {len(agent_out.reasoning_trace)}')

## 2. Synthesize Peer-Reviewed Astronomical Report

In [3]:
rep_gen = AstronomicalReportGenerator()
report = rep_gen.generate(
    target_id='Kepler-10',
    classification=agent_out.final_classification,
    confidence=agent_out.final_confidence,
    energy_score=agent_out.energy_score,
    is_anomaly=agent_out.is_anomaly,
    simbad_data=agent_out.simbad_match.model_dump() if agent_out.simbad_match else {},
    ads_papers=agent_out.ads_match.model_dump().get('papers', []) if agent_out.ads_match else [],
    astronomical_notes=agent_out.astronomical_notes
)

print('--- EXECUTIVE SUMMARY ---')
print(report.executive_summary)
print('\n--- RECOMMENDED OBSERVATIONAL FOLLOW-UP ---')
for f in report.recommended_followup:
    print(' *', f)